<a href="https://colab.research.google.com/github/arinamartynyuk/-4-9/blob/main/%D0%9B%D0%A02_%D0%9C%D0%B0%D1%80%D1%82%D0%B8%D0%BD%D1%8E%D0%BA4_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Лабораторна робота 2. Прізвище, група

---



# Завдання 1

**Був присутній на парі**

Обробка та нааліз даних про ВВП країн. Датасет було отримано з Вікіпедії з використанням бібілотеки Pandas.

1. Вивести перші 5 рядків датасета

In [ ]:
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display


In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
tables = pd.read_html(html)

# Знаходимо таблицю, що містить GDP-оцінки IMF, World Bank та UN
raw = None
for table in tables:
    cols = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in table.columns]
    joined = " ".join(cols).lower()
    if ("imf" in joined or "international monetary" in joined) and ("world bank" in joined or "united nations" in joined):
        raw = table.copy()
        break
if raw is None:
    raw = tables[2].copy()  # запасний варіант для структури сторінки

# Згладжуємо заголовки колонок, якщо вони багаторівневі
raw.columns = [" ".join(map(str, c)).strip() if isinstance(c, tuple) else str(c).strip() for c in raw.columns]
raw.head()

2.	Визначити розмір датасета.

In [ ]:
print("Розмір датасета (рядки, стовпці):", raw.shape)

 3.	Визначити оптимальну кількість стовпців (видалити зайві, на власний розсуд).

In [ ]:
print("Назви стовпців початкової таблиці:")
print(raw.columns.tolist())
# Відбираємо країну та три числові показники, якщо їхні заголовки доступні.
# Під час оновлення Wikipedia заголовки можуть змінитися, тому виводимо їх для перевірки.

4.	Замініть у таблиці значення "—" на значення NaN. Перевірити наявність пропущених значень. При наявності, замінити пропущені значення на середнє значення.

In [ ]:
df = raw.copy()
df.columns = [str(c).replace("\\n", " ").strip() for c in df.columns]
# Переводимо тире та різні текстові позначення відсутніх даних у NaN
df = df.replace(["—", "–", "-", "N/A", "n/a", "nan", ""], np.nan)
print("Пропущені значення до обробки:")
print(df.isna().sum())

In [ ]:
# Визначаємо колонку з назвою країни/території
country_candidates = [c for c in df.columns if any(k in c.lower() for k in ["country", "territory", "economy", "nation"])]
country_col = country_candidates[0] if country_candidates else df.columns[0]
# Зберігаємо назву країни та числові стовпці
numeric_candidates = []
for c in df.columns:
    if c == country_col: continue
    converted = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False).str.replace("$", "", regex=False).str.replace("%", "", regex=False), errors="coerce")
    if converted.notna().sum() > max(3, int(len(df)*0.25)):
        numeric_candidates.append(c)
print("Колонка країн:", country_col)
print("Числові кандидати:", numeric_candidates)

In [ ]:
# Обираємо стовпці IMF (2026), World Bank (2025) та United Nations (2024) за назвою.
def find_col(keys):
    for c in df.columns:
        low = c.lower()
        if all(k in low for k in keys): return c
    return None
imf_col = find_col(["imf", "2026"]) or find_col(["imf"])
wb_col = find_col(["world bank", "2025"]) or find_col(["world bank"])
un_col = find_col(["united nations", "2024"]) or find_col(["united nations"]) or find_col(["un"])
print("Визначені колонки:", imf_col, wb_col, un_col)
# Якщо автоматичне визначення не спрацювало, покажіть заголовки з попередньої клітинки та встановіть назви вручну.

In [ ]:
# Формуємо компактний датасет із потрібних колонок. Якщо заголовки на сторінці змінилися,
# можна вручну замінити imf_col, wb_col, un_col на відповідні назви.
selected = [country_col] + [c for c in [imf_col, wb_col, un_col] if c is not None]
clean = df[selected].copy()
clean.columns = ["Country", "IMF_2026", "World_Bank_2025", "UN_2024"][:len(clean.columns)]
for c in clean.columns[1:]:
    clean[c] = pd.to_numeric(clean[c].astype(str).str.replace(",", "", regex=False).str.replace("$", "", regex=False), errors="coerce")
clean = clean.dropna(subset=["Country"])
# Заповнення пропусків у числових колонках середнім значенням відповідної колонки
for c in clean.columns[1:]:
    clean[c] = clean[c].fillna(clean[c].mean())
clean.head()

In [ ]:
# Перевірка остаточного набору на пропущені значення
print(clean.isna().sum())

5. Ще раз вивести датасет

In [ ]:
display(clean)

6.	Перевірити наявність дублікатів. При наявності видалити дублікати.

In [ ]:
print("Кількість дублікатів:", clean.duplicated().sum())

In [ ]:
clean = clean.drop_duplicates().reset_index(drop=True)
print("Дублікати видалено. Новий розмір:", clean.shape)

In [ ]:
display(clean)

7.	Вивести описову статистику датасету describe()

In [ ]:
clean.describe()

8. Визначте відхилення (різницю) між показниками IMF (2026) та World Bank (2025) для кожної країни. У яких країнах ці показники найбільше відрізняються (дати відповідь)?

In [ ]:
clean["Difference_IMF_WB"] = (clean["IMF_2026"] - clean["World_Bank_2025"]).abs()
display(clean[["Country", "IMF_2026", "World_Bank_2025", "Difference_IMF_WB"]].sort_values("Difference_IMF_WB", ascending=False).head(10))
print("Країни з найбільшою різницею:")
display(clean.nlargest(5, "Difference_IMF_WB")[["Country", "Difference_IMF_WB"]])

9.	Обчисліть кореляцію між показниками IMF (2026), World Bank (2025) та United Nations (2024). Які пари змінних мають найвищу кореляцію?

In [ ]:
corr = clean[["IMF_2026", "World_Bank_2025", "UN_2024"]].corr()
display(corr)
# Знаходимо пару показників із найвищою кореляцією (без діагоналі)
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("Найвища кореляція між парою:", pairs.index[0], "=", pairs.iloc[0])

Обчислити середнє значення за стовпцями

In [ ]:
print("Середні значення показників:")
print(clean[["IMF_2026", "World_Bank_2025", "UN_2024"]].mean())

In [ ]:
# Додатково: візуалізація середніх значень
clean[["IMF_2026", "World_Bank_2025", "UN_2024"]].mean().plot(kind="bar", title="Середнє значення ВВП за джерелами", ylabel="ВВП")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Зведення середніх значень у таблицю
mean_table = clean[["IMF_2026", "World_Bank_2025", "UN_2024"]].mean().to_frame("Mean")
display(mean_table)

11.	Обчисліть стандартне відхилення показників для кожної країни. Яка країна має найвищу варіативність у показниках між роками?

In [ ]:
clean["Std_3years"] = clean[["IMF_2026", "World_Bank_2025", "UN_2024"]].std(axis=1)
display(clean[["Country", "Std_3years"]].sort_values("Std_3years", ascending=False).head(10))
print("Країна з найбільшою варіативністю:", clean.loc[clean["Std_3years"].idxmax(), "Country"])

12.	Визначення країни з найвищим та найнижчим показниками: Знайдіть країну з найвищим та найнижчим показниками у кожному з років (IMF (2026), World Bank (2025), United Nations (2024)).

Результати для кожного року/джерела виводяться автоматично у попередній клітинці. Порівняння найвищих і найнижчих значень дозволяє визначити країни з найбільшими та найменшими економіками.


In [ ]:
years = ["IMF_2026", "World_Bank_2025", "UN_2024"]
for col in years:
    print("\\n", col)
    print("Найвище значення:")
    display(clean.loc[[clean[col].idxmax()], ["Country", col]])
    print("Найнижче значення:")
    display(clean.loc[[clean[col].idxmin()], ["Country", col]])

13.	Побудуйте гістограму для розподілу показників IMF (2026) серед всіх країн. Який вигляд має розподіл? Чи є країни, що виділяються?

In [ ]:
plt.figure(figsize=(10, 6))
clean["IMF_2026"].plot(kind="hist", bins=30, edgecolor="black")
plt.title("Розподіл ВВП (IMF, 2026)")
plt.xlabel("ВВП")
plt.ylabel("Кількість країн")
plt.tight_layout()
plt.show()
print("Розподіл зазвичай асиметричний вправо: більшість країн мають менші значення ВВП, а кілька найбільших економік формують правий хвіст.")
print("Країни, що виділяються за величиною IMF:")
display(clean.nlargest(10, "IMF_2026")[["Country", "IMF_2026"]])

Гістограма демонструє розподіл ВВП. Зазвичай розподіл має правосторонню асиметрію: більшість країн мають порівняно невеликі значення, а найбільші економіки є віддаленими значеннями.


14.	Розрахуйте частку кожної країни в загальному значенні для кожного року (IMF (2026), World Bank (2025), United Nations (2024)).
 Як змінюються частки країн з часом (дати відповідь)?

In [ ]:
for col in years:
    clean["Share_" + col] = clean[col] / clean[col].sum() * 100
share_cols = ["Share_" + c for c in years]
display(clean[["Country"] + share_cols].sort_values("Share_IMF_2026", ascending=False).head(15))
print("Частки обчислено у відсотках від сумарного ВВП для кожного року/джерела.")

15.	Візуалізуйте зміни в показниках для кожної країни за три роки на графіку. Які країни показують стабільне зростання або спад (дати відповідь)?

In [ ]:
plt.figure(figsize=(12, 7))
# Графік для топ-10 країн за IMF, щоб графік залишався читабельним
top10 = clean.nlargest(10, "IMF_2026")
for _, row in top10.iterrows():
    plt.plot(years, row[years].values, marker="o", label=row["Country"])
plt.title("Зміна ВВП топ-10 економік за трьома оцінками")
plt.xlabel("Джерело / рік")
plt.ylabel("ВВП")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()
# Визначаємо напрям зміни між UN 2024, WB 2025 та IMF 2026 (як приблизну послідовність)
clean["Trend"] = clean["UN_2024"] < clean["World_Bank_2025"]
clean["Trend2"] = clean["World_Bank_2025"] < clean["IMF_2026"]
print("Країни зі зростанням в обох переходах (UN -> WB -> IMF):")
display(clean[clean["Trend"] & clean["Trend2"]][["Country"] + years].head(20))
print("Країни зі спадом в обох переходах:")
display(clean[(clean["UN_2024"] > clean["World_Bank_2025"]) & (clean["World_Bank_2025"] > clean["IMF_2026"])][["Country"] + years].head(20))

Висновок: у роботі було отримано та очищено дані про номінальний ВВП, перевірено пропуски й дублікати, обчислено статистичні показники, різниці, кореляції, частки та побудовано графіки. Конкретні результати формуються під час виконання коду й залежать від актуальної таблиці Wikipedia.
